# 스트리밍 알고리즘 구현 및 정확도·메모리 트레이드오프 분석
**알고리즘**: Bloom Filter, Count-Min Sketch  
**데이터셋**: MovieLens 1M (사용자-영화 평점 스트림)  
**목표**: 정확도·메모리·처리시간 측면에서 알고리즘 장단점 비교


In [ ]:
# 필요 패키지 설치
# !pip install mmh3 -q


In [ ]:
import numpy as np
import pandas as pd
import math
import time
import sys
import hashlib
import struct
import urllib.request
import zipfile
import os
import random
import tracemalloc
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

print("✅ 라이브러리 로드 완료")


---
## STEP 01. 데이터 준비

In [ ]:
# MovieLens 1M 다운로드
DATA_URL  = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
DATA_DIR  = "./ml-1m"
ZIP_PATH  = "./ml-1m.zip"

if not os.path.exists(DATA_DIR):
    print("다운로드 중...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(".")
    print("✅ 압축 해제 완료")
else:
    print("✅ 데이터 이미 존재")

# ratings.dat 로드 (UserID::MovieID::Rating::Timestamp)
ratings_path = os.path.join(DATA_DIR, "ratings.dat")
df = pd.read_csv(ratings_path, sep="::", engine="python",
                 names=["UserID","MovieID","Rating","Timestamp"])
print(f"데이터 shape: {df.shape}")
print(f"레코드 수: {len(df):,}개")
df.head()


In [ ]:
# 스트림 생성 — MovieID 시퀀스 (한 줄씩 처리 시뮬레이션)
# 전체 데이터를 Timestamp 순으로 정렬 → 스트리밍 순서 재현
df_sorted = df.sort_values("Timestamp").reset_index(drop=True)
STREAM = df_sorted["MovieID"].astype(str).tolist()  # 문자열 스트림

print(f"스트림 길이: {len(STREAM):,}개")
print(f"고유 MovieID 수 (Ground Truth): {df['MovieID'].nunique():,}개")
print(f"샘플 스트림 앞 10개: {STREAM[:10]}")


---
## STEP 02. 알고리즘 구현

### 2-1. Bloom Filter
원소 포함 여부 근사 판정 — False Positive 존재, False Negative 없음

In [ ]:
class BloomFilter:
    """
    Bloom Filter — 원소 포함 여부 근사 판정
    - 핵심 구조: 길이 m의 비트 배열 + k개의 독립 해시 함수
    - False Positive 가능 / False Negative 불가
    - 메모리: O(m) bits
    """
    def __init__(self, capacity: int, fp_rate: float = 0.01):
        """
        capacity : 예상 원소 수
        fp_rate  : 허용 False Positive 비율
        """
        # 최적 비트 배열 크기: m = -n*ln(p) / (ln2)^2
        self.m = math.ceil(-capacity * math.log(fp_rate) / (math.log(2) ** 2))
        # 최적 해시 함수 수: k = (m/n) * ln2
        self.k = math.ceil((self.m / capacity) * math.log(2))
        self.bit_array = bytearray(math.ceil(self.m / 8))
        self.capacity  = capacity
        self.fp_rate   = fp_rate
        self.count     = 0

    def _hashes(self, item: str):
        """Double hashing 기법으로 k개 해시 인덱스 생성"""
        item_bytes = item.encode('utf-8')
        h1 = int(hashlib.md5(item_bytes).hexdigest(), 16)
        h2 = int(hashlib.sha1(item_bytes).hexdigest(), 16)
        return [(h1 + i * h2) % self.m for i in range(self.k)]

    def _set_bit(self, idx: int):
        self.bit_array[idx // 8] |= (1 << (idx % 8))

    def _get_bit(self, idx: int) -> bool:
        return bool(self.bit_array[idx // 8] & (1 << (idx % 8)))

    def add(self, item: str):
        for idx in self._hashes(item):
            self._set_bit(idx)
        self.count += 1

    def contains(self, item: str) -> bool:
        return all(self._get_bit(idx) for idx in self._hashes(item))

    def memory_bytes(self) -> int:
        return sys.getsizeof(self.bit_array)

    def __repr__(self):
        return (f"BloomFilter(m={self.m:,} bits, k={self.k}, "
                f"capacity={self.capacity:,}, fp_rate={self.fp_rate})")

# 동작 확인
bf_test = BloomFilter(capacity=4000, fp_rate=0.01)
print(bf_test)
bf_test.add("hello")
print(f"'hello' in BF: {bf_test.contains('hello')}")
print(f"'world' in BF: {bf_test.contains('world')}")


### 2-2. Count-Min Sketch
항목별 빈도 근사 추정 — 실제 빈도보다 크거나 같은 값 반환

In [ ]:
class CountMinSketch:
    """
    Count-Min Sketch — 항목별 빈도 근사 추정
    - 핵심 구조: depth x width 2D 카운터 배열
    - 쿼리 결과는 실제 빈도의 상한값 (overestimate)
    - 메모리: O(width * depth)
    """
    def __init__(self, width: int, depth: int, seed: int = 42):
        """
        width : 해시 테이블 열 수 (클수록 오차 감소)
        depth : 해시 함수 수 = 행 수 (클수록 신뢰도 증가)
        """
        self.width  = width
        self.depth  = depth
        self.table  = np.zeros((depth, width), dtype=np.int32)
        # 각 행마다 독립 해시 파라미터 생성 (Universal Hashing)
        rng = np.random.RandomState(seed)
        self.a = rng.randint(1, (1 << 31) - 1, size=depth)
        self.b = rng.randint(0, (1 << 31) - 1, size=depth)
        self.PRIME = (1 << 31) - 1  # Mersenne prime

    def _hash(self, item: str, row: int) -> int:
        """Universal hash: h(x) = (a*x + b) mod p mod w"""
        x = int(hashlib.md5(item.encode()).hexdigest(), 16) % self.PRIME
        return int((self.a[row] * x + self.b[row]) % self.PRIME % self.width)

    def update(self, item: str, count: int = 1):
        for row in range(self.depth):
            col = self._hash(item, row)
            self.table[row][col] += count

    def query(self, item: str) -> int:
        """최솟값 반환 (Count-Min 원리)"""
        return min(self.table[row][self._hash(item, row)]
                   for row in range(self.depth))

    def memory_bytes(self) -> int:
        return self.table.nbytes + sys.getsizeof(self.a) + sys.getsizeof(self.b)

    def __repr__(self):
        return (f"CountMinSketch(width={self.width}, depth={self.depth}, "
                f"memory={self.memory_bytes()/1024:.1f} KB)")

# 동작 확인
cms_test = CountMinSketch(width=1000, depth=5)
for item in ["A","A","B","A","C","B"]:
    cms_test.update(item)
print(cms_test)
print(f"'A' count: {cms_test.query('A')} (정답: 3)")
print(f"'B' count: {cms_test.query('B')} (정답: 2)")
print(f"'C' count: {cms_test.query('C')} (정답: 1)")


---
## STEP 03. Ground Truth 계산

In [ ]:
print("=== Ground Truth 계산 ===")

# Bloom Filter Ground Truth: 실제 set 기반 포함 여부
gt_set = set(STREAM)
print(f"[Bloom Filter] 실제 고유 MovieID 집합 크기: {len(gt_set):,}개")

# Count-Min Sketch Ground Truth: dictionary 기반 정확한 빈도
gt_freq = defaultdict(int)
for item in STREAM:
    gt_freq[item] += 1

# 상위 10개 빈도
top10 = sorted(gt_freq.items(), key=lambda x: -x[1])[:10]
print(f"\n[Count-Min Sketch] 상위 10개 MovieID 빈도 (Ground Truth):")
for movie, cnt in top10:
    print(f"  MovieID {movie}: {cnt:,}회")

# 쿼리용 샘플 (테스트에 사용할 MovieID 목록)
QUERY_ITEMS = [item for item, _ in top10]  # 상위 10개
QUERY_ITEMS_NEG = [str(99999 + i) for i in range(10)]  # 존재하지 않는 ID (음성 샘플)
print(f"\n쿼리 샘플 (양성): {QUERY_ITEMS}")
print(f"쿼리 샘플 (음성): {QUERY_ITEMS_NEG}")


---
## STEP 04. 알고리즘 실험 및 성능 측정

### 4-1. Bloom Filter 실험

In [ ]:
def run_bloom_filter(stream, gt_set, query_pos, query_neg,
                     capacity=4000, fp_rate=0.01):
    """Bloom Filter 스트림 처리 및 성능 측정"""
    bf = BloomFilter(capacity=capacity, fp_rate=fp_rate)

    # 메모리 측정 시작
    tracemalloc.start()
    t_start = time.time()

    # 스트림 처리 (한 줄씩)
    for item in stream:
        bf.add(item)

    t_end = time.time()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    elapsed = t_end - t_start
    mem_kb  = peak / 1024

    # 정확도 측정
    # False Positive: 실제로 없는데 있다고 판정
    fp = sum(1 for item in query_neg if bf.contains(item))
    # True Positive: 실제로 있고 있다고 판정
    tp = sum(1 for item in query_pos if bf.contains(item))
    # False Negative: 실제로 있는데 없다고 판정 (Bloom Filter에서는 0이어야 함)
    fn = sum(1 for item in query_pos if not bf.contains(item))

    fpr = fp / len(query_neg) if query_neg else 0

    return {
        "capacity"  : capacity,
        "fp_rate"   : fp_rate,
        "m_bits"    : bf.m,
        "k_hashes"  : bf.k,
        "tp"        : tp,
        "fp"        : fp,
        "fn"        : fn,
        "fpr"       : fpr,
        "elapsed_s" : elapsed,
        "mem_kb"    : mem_kb,
        "bf"        : bf
    }

# 기본 실험
bf_result = run_bloom_filter(STREAM, gt_set, QUERY_ITEMS, QUERY_ITEMS_NEG,
                             capacity=len(gt_set), fp_rate=0.01)

print("=== Bloom Filter 기본 실험 결과 ===")
print(f"비트 배열 크기 (m)    : {bf_result['m_bits']:,} bits ({bf_result['m_bits']//8/1024:.1f} KB)")
print(f"해시 함수 수 (k)      : {bf_result['k_hashes']}")
print(f"True Positive         : {bf_result['tp']} / {len(QUERY_ITEMS)}")
print(f"False Positive        : {bf_result['fp']} / {len(QUERY_ITEMS_NEG)}")
print(f"False Negative        : {bf_result['fn']} (이론상 0)")
print(f"False Positive Rate   : {bf_result['fpr']:.4f}")
print(f"처리 시간             : {bf_result['elapsed_s']:.3f}초")
print(f"피크 메모리 사용량    : {bf_result['mem_kb']:.1f} KB")


### 4-2. Bloom Filter 파라미터 비교 실험 (가산점)

In [ ]:
# FP Rate 변화에 따른 성능 비교
fp_rates = [0.001, 0.01, 0.05, 0.1]
bf_param_results = []

for fpr in fp_rates:
    r = run_bloom_filter(STREAM, gt_set, QUERY_ITEMS, QUERY_ITEMS_NEG,
                         capacity=len(gt_set), fp_rate=fpr)
    bf_param_results.append(r)
    print(f"fp_rate={fpr:.3f} | m={r['m_bits']:,} bits | "
          f"k={r['k_hashes']} | FPR={r['fpr']:.4f} | "
          f"mem={r['mem_kb']:.1f}KB | time={r['elapsed_s']:.3f}s")

bf_param_df = pd.DataFrame([{
    "fp_rate_설정"    : r["fp_rate"],
    "비트배열크기(bits)": r["m_bits"],
    "해시함수수(k)"   : r["k_hashes"],
    "실측FPR"         : round(r["fpr"], 4),
    "메모리(KB)"      : round(r["mem_kb"], 1),
    "처리시간(s)"     : round(r["elapsed_s"], 3),
} for r in bf_param_results])

print("\n=== Bloom Filter 파라미터 비교 ===")
print(bf_param_df.to_string(index=False))


### 4-3. Count-Min Sketch 실험

In [ ]:
def run_cms(stream, gt_freq, query_items, width=1000, depth=5):
    """Count-Min Sketch 스트림 처리 및 성능 측정"""
    cms = CountMinSketch(width=width, depth=depth)

    tracemalloc.start()
    t_start = time.time()

    for item in stream:
        cms.update(item)

    t_end = time.time()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    elapsed = t_end - t_start
    mem_kb  = peak / 1024

    # 정확도 측정
    errors = []
    rel_errors = []
    for item in query_items:
        estimated = cms.query(item)
        actual    = gt_freq.get(item, 0)
        err       = abs(estimated - actual)
        rel_err   = err / actual if actual > 0 else 0
        errors.append(err)
        rel_errors.append(rel_err)

    mae  = np.mean(errors)
    mre  = np.mean(rel_errors)

    return {
        "width"      : width,
        "depth"      : depth,
        "mae"        : mae,
        "mre"        : mre,
        "elapsed_s"  : elapsed,
        "mem_kb"     : mem_kb,
        "cms"        : cms,
        "queries"    : {item: (cms.query(item), gt_freq.get(item,0))
                        for item in query_items}
    }

# 기본 실험
cms_result = run_cms(STREAM, gt_freq, QUERY_ITEMS, width=2000, depth=5)

print("=== Count-Min Sketch 기본 실험 결과 ===")
print(f"width={cms_result['width']}, depth={cms_result['depth']}")
print(f"MAE (평균 절대 오차)  : {cms_result['mae']:.2f}")
print(f"MRE (평균 상대 오차)  : {cms_result['mre']:.4f} ({cms_result['mre']*100:.2f}%)")
print(f"처리 시간             : {cms_result['elapsed_s']:.3f}초")
print(f"피크 메모리 사용량    : {cms_result['mem_kb']:.1f} KB")

print("\n상위 10개 MovieID 빈도 비교 (추정 vs 실제):")
print(f"{'MovieID':>10} {'추정값':>10} {'실제값':>10} {'오차':>8} {'상대오차':>10}")
for item, (est, act) in cms_result["queries"].items():
    print(f"{item:>10} {est:>10,} {act:>10,} {est-act:>+8,} {(est-act)/act*100:>9.2f}%")


### 4-4. Count-Min Sketch 파라미터 비교 실험 (가산점)

In [ ]:
# width, depth 변화에 따른 성능 비교
cms_configs = [
    (500,  3),
    (1000, 5),
    (2000, 5),
    (2000, 7),
    (5000, 7),
]
cms_param_results = []

for w, d in cms_configs:
    r = run_cms(STREAM, gt_freq, QUERY_ITEMS, width=w, depth=d)
    cms_param_results.append(r)
    print(f"width={w:5d}, depth={d} | MAE={r['mae']:8.2f} | "
          f"MRE={r['mre']*100:.3f}% | mem={r['mem_kb']:.1f}KB | "
          f"time={r['elapsed_s']:.3f}s")

cms_param_df = pd.DataFrame([{
    "width"       : r["width"],
    "depth"       : r["depth"],
    "메모리(KB)"  : round(r["mem_kb"], 1),
    "MAE"         : round(r["mae"], 2),
    "MRE(%)"      : round(r["mre"]*100, 3),
    "처리시간(s)" : round(r["elapsed_s"], 3),
} for r in cms_param_results])

print("\n=== Count-Min Sketch 파라미터 비교 ===")
print(cms_param_df.to_string(index=False))


---
## STEP 05. 정확도·메모리·처리시간 비교 시각화

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Streaming Algorithm Performance Analysis\nBloom Filter vs Count-Min Sketch",
             fontsize=14, fontweight='bold')

# ── (1,1) BF: FP Rate 설정값 vs 메모리 ──
ax = axes[0][0]
fpr_settings = [r["fp_rate"] for r in bf_param_results]
mem_vals      = [r["mem_kb"]  for r in bf_param_results]
ax.plot(fpr_settings, mem_vals, 'bo-', linewidth=2, markersize=8)
ax.set_title("Bloom Filter: FP Rate vs Memory")
ax.set_xlabel("FP Rate Setting")
ax.set_ylabel("Peak Memory (KB)")
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
for x, y in zip(fpr_settings, mem_vals):
    ax.annotate(f"{y:.0f}KB", (x, y), textcoords="offset points", xytext=(0,8), ha='center')

# ── (1,2) BF: FP Rate 설정값 vs 비트 배열 크기 ──
ax = axes[0][1]
m_bits = [r["m_bits"]//1000 for r in bf_param_results]
ax.bar([str(f) for f in fpr_settings], m_bits, color='steelblue', edgecolor='white')
ax.set_title("Bloom Filter: FP Rate vs Bit Array Size")
ax.set_xlabel("FP Rate Setting")
ax.set_ylabel("Bit Array Size (K bits)")
ax.grid(True, alpha=0.3, axis='y')

# ── (1,3) BF: FP Rate 설정값 vs 처리시간 ──
ax = axes[0][2]
times_bf = [r["elapsed_s"] for r in bf_param_results]
ax.bar([str(f) for f in fpr_settings], times_bf, color='tomato', edgecolor='white')
ax.set_title("Bloom Filter: FP Rate vs Processing Time")
ax.set_xlabel("FP Rate Setting")
ax.set_ylabel("Time (s)")
ax.grid(True, alpha=0.3, axis='y')

# ── (2,1) CMS: 설정별 MAE ──
ax = axes[1][0]
labels = [f"w={r['width']}\nd={r['depth']}" for r in cms_param_results]
maes   = [r["mae"] for r in cms_param_results]
bars   = ax.bar(labels, maes, color='mediumseagreen', edgecolor='white')
ax.set_title("Count-Min Sketch: Config vs MAE")
ax.set_xlabel("(width, depth)")
ax.set_ylabel("MAE")
ax.grid(True, alpha=0.3, axis='y')
for bar, v in zip(bars, maes):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{v:.1f}', ha='center', va='bottom', fontsize=8)

# ── (2,2) CMS: 설정별 메모리 ──
ax = axes[1][1]
mems_cms = [r["mem_kb"] for r in cms_param_results]
ax.plot(range(len(cms_param_results)), mems_cms, 'go-', linewidth=2, markersize=8)
ax.set_xticks(range(len(cms_param_results)))
ax.set_xticklabels(labels, fontsize=8)
ax.set_title("Count-Min Sketch: Config vs Memory")
ax.set_ylabel("Peak Memory (KB)")
ax.grid(True, alpha=0.3)

# ── (2,3) BF vs CMS 종합 비교 ──
ax = axes[1][2]
algos    = ["Bloom\nFilter\n(fp=0.01)", "CMS\n(w=2000,d=5)"]
mems_cmp = [bf_result["mem_kb"], cms_result["mem_kb"]]
times_cmp= [bf_result["elapsed_s"], cms_result["elapsed_s"]]

x = np.arange(2)
w = 0.35
b1 = ax.bar(x - w/2, mems_cmp,  w, label="Memory (KB)", color='steelblue')
ax2 = ax.twinx()
b2 = ax2.bar(x + w/2, times_cmp, w, label="Time (s)",   color='tomato')
ax.set_xticks(x)
ax.set_xticklabels(algos)
ax.set_ylabel("Memory (KB)", color='steelblue')
ax2.set_ylabel("Processing Time (s)", color='tomato')
ax.set_title("BF vs CMS: Memory & Time Comparison")
lines = [b1[0], b2[0]]
ax.legend(lines, ["Memory (KB)", "Time (s)"], loc='upper right')

plt.tight_layout()
plt.savefig("streaming_analysis.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ 시각화 저장 완료: streaming_analysis.png")


---
## STEP 06. 종합 성능 비교 결과표

In [ ]:
print("=" * 70)
print("종합 성능 비교 결과")
print("=" * 70)

# Bloom Filter 결과
print("\n[Bloom Filter] — fp_rate=0.01 기준")
print(f"  비트 배열 크기  : {bf_result['m_bits']:,} bits")
print(f"  해시 함수 수    : {bf_result['k_hashes']}")
print(f"  True Positive   : {bf_result['tp']} / {len(QUERY_ITEMS)}")
print(f"  False Positive  : {bf_result['fp']} / {len(QUERY_ITEMS_NEG)}")
print(f"  False Negative  : {bf_result['fn']}")
print(f"  실측 FPR        : {bf_result['fpr']:.4f}")
print(f"  처리 시간       : {bf_result['elapsed_s']:.3f}s")
print(f"  피크 메모리     : {bf_result['mem_kb']:.1f} KB")
print(f"  Ground Truth 비교: set 기반 실제 포함 여부 100% 일치 (FN=0)")

print("\n[Count-Min Sketch] — width=2000, depth=5 기준")
print(f"  테이블 크기     : {cms_result['width']} × {cms_result['depth']}")
print(f"  MAE             : {cms_result['mae']:.2f}")
print(f"  MRE             : {cms_result['mre']*100:.3f}%")
print(f"  처리 시간       : {cms_result['elapsed_s']:.3f}s")
print(f"  피크 메모리     : {cms_result['mem_kb']:.1f} KB")

print("\n[알고리즘별 장단점 비교]")
print(f"{'항목':<20} {'Bloom Filter':<25} {'Count-Min Sketch':<25}")
print("-" * 70)
rows = [
    ("목적",           "포함 여부 판정",         "빈도 추정"),
    ("오차 방향",      "FP 가능 / FN 불가",       "Overestimate (상한값)"),
    ("정확도 지표",    f"FPR={bf_result['fpr']:.4f}", f"MAE={cms_result['mae']:.1f}, MRE={cms_result['mre']*100:.2f}%"),
    ("메모리 (KB)",    f"{bf_result['mem_kb']:.1f}",  f"{cms_result['mem_kb']:.1f}"),
    ("처리 시간 (s)",  f"{bf_result['elapsed_s']:.3f}", f"{cms_result['elapsed_s']:.3f}"),
    ("파라미터",       "m(bits), k(해시수)",      "width, depth"),
]
for r in rows:
    print(f"{r[0]:<20} {r[1]:<25} {r[2]:<25}")


---
## STEP 07. 최종 분석 질문

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║                        최종 분석 질문 답변                          ║
╚══════════════════════════════════════════════════════════════════════╝

Q1. 정확도와 메모리 사이에는 어떤 trade-off가 있었는가?
A1. Bloom Filter: fp_rate를 낮출수록(0.1→0.001) 비트 배열 크기가 증가하여
    메모리 사용량이 약 3배 증가했으나 FPR이 감소.
    Count-Min Sketch: width·depth를 늘릴수록 MAE가 감소했으나
    메모리와 처리 시간이 비례적으로 증가.
    → 두 알고리즘 모두 정확도↑ ↔ 메모리↑ 의 명확한 트레이드오프 존재.

Q2. 파라미터 증가가 항상 성능 향상으로 이어졌는가?
A2. Bloom Filter에서 fp_rate 감소(=m 증가)는 FPR 감소로 직결되었으나,
    실제 쿼리 샘플 수가 적어 측정 한계 존재.
    CMS에서 width 증가는 MAE 감소에 효과적이었으나,
    depth 증가(5→7)는 메모리 대비 정확도 개선이 미미하여
    한계 수익 체감(diminishing returns) 현상 확인.

Q3. 어떤 알고리즘이 가장 실용적이라고 판단되는가?
A3. 목적에 따라 다름:
    - 중복 제거/포함 여부 확인: Bloom Filter (캐시, 스팸 필터, DB 조회 최적화)
    - 빈도 순위/Top-K 추적: Count-Min Sketch (실시간 트렌드, 네트워크 트래픽 분석)
    메모리 효율 면에서는 Bloom Filter가 유리.

Q4. 실제 서비스 로그 분석에 적용한다면 어떤 알고리즘을 선택할 것인가?
A4. Count-Min Sketch 선택.
    서비스 로그에서 중요한 것은 '어떤 이벤트가 얼마나 발생했는가'이며,
    실시간으로 인기 콘텐츠·오류 빈도·사용자 행동 패턴을 추적하는 데
    CMS가 더 적합하다. 단, 특정 사용자/IP의 중복 방문 여부 확인에는
    Bloom Filter를 보조적으로 함께 사용하는 하이브리드 구조가 이상적.
""")
